# Lecture 6 Lab — Model Evaluation, Hyperparameter Tuning 
### Machine Learning for Robotics & Industrial Automation

| | |
|---|---|
| **Estimated time** | 3–4 hours  |
| **Tools** | Python, NumPy, pandas, matplotlib, scikit-learn |
| **Submit** | This completed notebook  |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID


### Important cell below 

You must assign 3 last digits of your student ID to this <code>s_id</code> variable. For example, if your student ID is 6710546988, you must assign
```python
s_id = 988
```
This code will be printed in later cells. **Each output cell that shows incorrect s_id printout will get -1 penalty per cell.**

s_id may be used in some part of code as well, such as setting random seed. So the output for each student may be slightly different. 

In [14]:
s_id = None  # replace with 3 last digits of your student ID.

## 1. Learning Objectives

By the end of this lab, you should be able to:

- Demonstrate why a single train/test split gives an unreliable estimate of model quality.
- Cross-validate a full preprocessing + model pipeline and report mean ± std.
- Read learning curves and validation curves to diagnose under/overfitting.
- Tune hyperparameters with `GridSearchCV` without ever touching the test set.
- Report final results the defensible way: held-out test score, confusion matrix, precision/recall, and ROC/AUC.

## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, cross_val_score, StratifiedKFold,
                                     learning_curve, validation_curve, GridSearchCV)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report, ConfusionMatrixDisplay,
                             roc_curve, roc_auc_score)

print("Environment OK")

## 3. Part A — The Dataset (Familiar Territory)

We reuse the Week 2 multiclass fault dataset — missing values, a categorical column, four classes — because this week is about *how you evaluate*, not about new data. The pipeline you built in Week 2 comes along for the ride.

In [ ]:
def generate_fault_dataset(n_per_class=150, seed=0):
    rng = np.random.default_rng(seed)
    class_names = ["Normal", "Bearing Fault", "Misalignment", "Imbalance"]
    centers = {
        "Normal":        dict(mean=0.02, std=0.15, rms=0.20, ptp=0.60),
        "Bearing Fault": dict(mean=0.03, std=0.35, rms=0.45, ptp=1.80),
        "Misalignment":  dict(mean=0.25, std=0.20, rms=0.55, ptp=0.90),
        "Imbalance":     dict(mean=0.05, std=0.18, rms=0.70, ptp=0.85),
    }
    rows = []
    for label_idx, cname in enumerate(class_names):
        c = centers[cname]
        for _ in range(n_per_class):
            rows.append({
                "mean": rng.normal(c["mean"], 0.05),
                "std": rng.normal(c["std"], 0.04),
                "rms": rng.normal(c["rms"], 0.06),
                "ptp": rng.normal(c["ptp"], 0.15),
                "machine_type": rng.choice(["CNC", "Press", "Lathe"]),
                "label": label_idx,
            })
    df = pd.DataFrame(rows)
    for col in ["std", "rms"]:
        missing_idx = rng.choice(df.index, size=int(0.05 * len(df)), replace=False)
        df.loc[missing_idx, col] = np.nan
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


df = generate_fault_dataset(seed=s_id) # note that s_id would cause each individual work to vary slightly
numeric_features = ["mean", "std", "rms", "ptp"]
categorical_features = ["machine_type"]
X = df[numeric_features + categorical_features]
y = df["label"]

print("Student ID : "+str(s_id))

# The Week 2 pipeline, verbatim
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])
pipe = Pipeline(steps=[("preprocess", preprocessor),
                       ("clf", RandomForestClassifier(random_state=0))])
print("Pipeline ready")

## 4. Part B — The Problem with One Split

Run the exact same pipeline on five different random splits. Watch the "model quality" number move even though nothing about the model changed.

In [ ]:
print("Student ID : "+str(s_id))
for seed in range(5):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    pipe.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, pipe.predict(X_te))
    print(f"random_state={seed}: accuracy = {acc:.3f}")

Which of those numbers would you put in a report to a plant manager? None of them individually — that's the point.

## 5. Part C — Cross-Validate the Whole Pipeline

Fill in the `TODO` below. Two things matter:

1. Cross-validate the **entire pipeline** — never preprocess first and cross-validate after, or your imputer/scaler leak information across folds.
2. Use `StratifiedKFold` so every fold has the same class balance.

In [ ]:
# TODO: use StratifiedKFold for cv with 5 splits, shuffled, and use your s_id as random_state 
cv = None
print("Student ID : "+str(s_id))
# TODO: use cross_val_score to evaluate `pipe` on X, y with cv=cv and
# scoring="f1_macro". Store the five scores in `cv_scores`.
cv_scores = None



print("Fold scores :", np.round(cv_scores, 3))
print(f"Mean ± std   : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

> That `mean ± std` line is what belongs in a report. It says both how good the model is *and* how sure you are.

## 6. Part D — Learning Curve: Would More Data Help?

In [ ]:
print("Student ID : "+str(s_id))
train_sizes, train_scores, val_scores = learning_curve(
    pipe, X, y, cv=cv, scoring="f1_macro",
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1,
)

plt.figure(figsize=(7, 4.5))
plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", color="#3E5C76", label="training score")
plt.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                 train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15, color="#3E5C76")
plt.plot(train_sizes, val_scores.mean(axis=1), marker="o", color="#FF6A39", label="validation score")
plt.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                 val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.15, color="#FF6A39")
plt.xlabel("Training set size"); plt.ylabel("F1 (macro)"); plt.legend()
plt.title("Learning curve — Week 2 pipeline on the fault dataset")
plt.tight_layout()
plt.show()

Are the curves converging? Is there a persistent gap? Note what you see — you'll interpret it in the reflection questions.

## 7. Part E — Validation Curve: Sweeping One Hyperparameter

Sweep the random forest's `max_depth` and watch training vs. validation score. This is Week 3's polynomial-degree sweep, generalized.

In [ ]:
depths = [2, 3, 4, 6, 8, 12, 16]
train_scores, val_scores = validation_curve(
    pipe, X, y, param_name="clf__max_depth", param_range=depths,
    cv=cv, scoring="f1_macro", n_jobs=-1,
)
print("Student ID : "+str(s_id))
plt.figure(figsize=(7, 4.5))
plt.plot(depths, train_scores.mean(axis=1), marker="o", color="#3E5C76", label="training score")
plt.plot(depths, val_scores.mean(axis=1), marker="o", color="#FF6A39", label="validation score")
plt.xlabel("max_depth"); plt.ylabel("F1 (macro)"); plt.legend()
plt.title("Validation curve — random forest depth")
plt.tight_layout()
plt.show()

> Note the parameter name: `clf__max_depth`, not `max_depth`. The double underscore routes the parameter *through the pipeline* to the step named `clf` — you'll need this syntax again in the grid search.

## 8. Part F — Grid Search, Then One Honest Look at the Test Set

Now the full canonical workflow:

1. Split once into train and test. **The test set goes in a drawer.**
2. `GridSearchCV` on the training set only (it cross-validates internally).
3. Take the best model out, evaluate it on the test set exactly once.

Fill in the `TODO` below.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=s_id, stratify=y)

param_grid = {
    "clf__max_depth": [4, 6, 8],
    "clf__n_estimators": [100, 200],
}
print("Student ID : "+str(s_id))
# TODO: create a GridSearchCV over `pipe` with param_grid, cv=cv,
# scoring="f1_macro", and fit it on X_train, y_train. Store it in `search`.
search = None


print("Best params        :", search.best_params_)
print(f"Best CV score      : {search.best_score_:.3f}")

In [ ]:
# The one and only look at the test set:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print("Student ID : "+str(s_id))
class_names = ["Normal", "Bearing Fault", "Misalignment", "Imbalance"]
print(classification_report(y_test, y_pred, target_names=class_names))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=class_names,
                                        cmap="Blues", xticks_rotation=20)
plt.title("Held-out test set — tuned pipeline")
plt.tight_layout()
plt.show()

## 9. Part G — ROC & AUC on the Safety-Relevant Question

For the line-stop decision, what matters is binary: **fault (any kind) vs. normal**. Collapse the labels and evaluate the ranking quality of the tuned model's probabilities.

In [ ]:
# Binary ground truth: 0 = Normal, 1 = any fault
y_test_binary = (y_test != 0).astype(int)
print("Student ID : "+str(s_id))
# P(fault) = 1 - P(Normal). Column 0 of predict_proba corresponds to class 0.
proba_fault = 1 - best_model.predict_proba(X_test)[:, 0]

fpr, tpr, thresholds = roc_curve(y_test_binary, proba_fault)
auc = roc_auc_score(y_test_binary, proba_fault)

plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, color="#FF6A39", linewidth=2, label=f"tuned model (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], color="#6B7A8F", linestyle="--", linewidth=1.25, label="chance (AUC = 0.5)")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.legend(); plt.title("ROC — fault vs. normal")
plt.tight_layout()
plt.show()

## 10. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. In Part B, how much did accuracy move across the five random splits? What would have gone wrong if you'd reported only the best of those five numbers?**

*Your answer:* 

**2. Interpret your learning curve from Part D: would collecting more data meaningfully improve this model? How do you know?**

*Your answer:* 

**3. Why must the imputer and scaler be cross-validated inside the pipeline, rather than fit once on all the data before cross-validation? What is the mistake you'd otherwise be making?**

*Your answer:*


**4. In a general assembly line, which error is more costly — a false alarm or a missed detection? What does that imply about where you should sit on the precision/recall trade-off?**

*Your answer:*


## 11. Deliverables & Submission

- This lab notebook, completed and able to run top-to-bottom without errors (`Kernel → Restart & Run All`).
- Written answers to the four reflection questions.

Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

## 13. Grading Rubric Guide

| Component | Weight |
|---|---|
| One-split problem reproduced and cross-validation correctly applied | 25% |
| Learning & validation curves generated and interpreted | 20% |
| Grid search correctly built; test set touched exactly once | 30% |
| ROC/AUC analysis and reflection questions | 15% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 10% |



---
**Next week:** Part 2 of the course begins — *From Neural Network Foundations to PyTorch*: tensors, autograd, and your first neural network.

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>